## Assignment Week 7

## Gyan Kannur 

### GPT-2 Fine-Tuning with LoRA

https://blog.devgenius.io/sculpting-language-gpt-2-fine-tuning-with-lora-1caf3bfbc3c6

## The Mission

The task will be to fine-tune GPT 2 (https://huggingface.co/openai-community/gpt2) on this quote tagging database https://huggingface.co/datasets/Abirate/english_quotes to allow our new model to preform a quote tagging task for us.

First lets run the model (in this case GPT-2) from hugging face with the default weights
 
initial prompt: Life is like a box of chocolates, you never know what you are gonna get


In [7]:
MODEL="gpt2"
initial_prompt="Life is like a box of chocolates, you never know what you are gonna get"

In [8]:
import torch
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    device_map='auto',
)

config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)
model = model.to(device)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token

with torch.no_grad():
    batch = tokenizer(f"{initial_prompt} ->: ", return_tensors='pt').to(device)
    output_tokens = model.generate(**batch, max_new_tokens=25)

print(f"{device =}")
print('\n\n', tokenizer.decode(output_tokens[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


device ='cpu'


 Life is like a box of chocolates, you never know what you are gonna get ->: ~~~

I'm not sure if I'm going to be able to get a good night's sleep, but I


### Observations

It seems GPT 2 thinks I am asking it to complete some script for a play. Let’s fix that with fine-tuning!

### Training

I ran the below in google-colab as my system did not have cuda support. 
Alternatively the author provided an already existing weights and skip below training and directly go to the inference.

My aim to try both


In [2]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = AutoModelForCausalLM.from_pretrained(
        MODEL,
    device_map='auto',
)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token

# FREEZE WEIGHTS
for param in model.parameters():
    param.requires_grad = False

# LoRa
config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

# LOAD AND STURCTURE DATA
data = load_dataset("Abirate/english_quotes")


def merge_columns(entry):
    entry["prediction"] = entry["quote"] + " ->: " + str(entry["tags"])
    return entry


data['train'] = data['train'].map(merge_columns)
print(data['train']['prediction'][:5])

data = data.map(lambda samples: tokenizer(samples['prediction']), batched=True)


def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )


print_trainable_parameters(model)

# TRAINING
trainer = transformers.Trainer(
    model=model,
    train_dataset=data['train'],
    args=transformers.TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=100,
        max_steps=500,
        learning_rate=2e-4,
        logging_steps=1,
        output_dir='outputs',
        auto_find_batch_size=True
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
model.config.use_cache = False
trainer.train()

torch.save(model.state_dict(), './data/lora_local.pt')

### Observations

When run using google colab, 

It complained about /usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.

Fix:
Go to Hugging Face Tokens Settings.
Click "New Token", give it a name, and set the role to "Write" (or "Read" if you only need to download models/datasets).
Copy the generated token.


Alternative: Set the Token as an Environment Variable

import os
from huggingface_hub import login

os.environ["HF_TOKEN"] = "your_huggingface_token_here"  # Replace with your token
login(token=os.environ["HF_TOKEN"])


#### Why This Matters

Faster downloads: Authenticated users get higher rate limits.
Access to private models/datasets: Required if you need to use gated or private resources.
Avoid warnings: Eliminates the unauthenticated request warnings.

Training loss 
This training took close to 10mins
It started off with a very high training loss of around 5+
After about 500 epochs, the training loss was 2.79. That is about 50% reduction, there is scope for further loss with the right set of hyperparameters.






#### Inference

I have created a function which can accept different hyper-parameters and intial prompt.
For this exercise, i am passing 2 different hyper-parameters, the one provided by the author and second one trained by me using google colab.

Let us see how the results differ or same


In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

def load_and_generate(
        model_name: str,
        lora_r: int = 16,
        lora_alpha: int = 32,
        lora_dropout: float = 0.05,
        checkpoint_path: str = "./data/lora-hyperparams.pt",
        initial_prompt: str = "",
        max_new_tokens: int = 25,
):
    """
    Load a LoRA-fine-tuned model and generate text.

    Args:
        model_name (str): Name of the base model (e.g., "gpt2").
        lora_r (int): LoRA rank. Default: 16.
        lora_alpha (int): LoRA alpha. Default: 32.
        lora_dropout (float): LoRA dropout. Default: 0.05.
        checkpoint_path (str): Path to the LoRA checkpoint. Default: "./data/lora-hyperparams.pt".
        initial_prompt (str): Input prompt for generation.
        max_new_tokens (int): Max tokens to generate. Default: 25.
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Load model and tokenizer
    model = AutoModelForCausalLM.from_pretrained(model_name, device_map='auto')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    # Configure LoRA
    config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )

    # Apply LoRA and load checkpoint
    model = get_peft_model(model, config)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.to(device)

    # Generate output
    with torch.no_grad():
        batch = tokenizer(f"{initial_prompt} ->: ", return_tensors='pt').to(device)
        output_tokens = model.generate(**batch, max_new_tokens=max_new_tokens)

    return tokenizer.decode(output_tokens[0], skip_special_tokens=True)



In [14]:

author_provided_weights ="./data/lora-hyperparams.pt"

output_custom = load_and_generate(
    model_name=MODEL,
    lora_r=32,
    lora_alpha=64,
    lora_dropout=0.1,
    checkpoint_path=author_provided_weights,
    initial_prompt=initial_prompt,
)
# print("\nGenerated output (custom hyperparams):", output_custom)
print(f"\n Generated Output for author provided weights {author_provided_weights} ")
print(output_custom)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



 Generated Output for author provided weights ./data/lora-hyperparams.pt 
Life is like a box of chocolates, you never know what you are gonna get ->: ~~~: ['chocolate', 'chocolate-dessert', 'chocolate-dessert-recipe'] ->


In [15]:

colab_trained_weights ="./data/lora-self.pt"

output_custom = load_and_generate(
    model_name=MODEL,
    lora_r=32,
    lora_alpha=64,
    lora_dropout=0.1,
    checkpoint_path=colab_trained_weights,
    initial_prompt=initial_prompt,
)
print(f"\n Generated Output for colab generated weights {colab_trained_weights} ")
print(output_custom)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



 Generated Output for colab generated weights ./data/lora-self.pt 
Life is like a box of chocolates, you never know what you are gonna get ->: ~~~I'm not sure what to say ->: 'I'm not sure what to say ->: 'I'm not



#### Observations

we have fine-tuned GPT2 to preform our specific task, this was done with a pretty low-powered GPU (using google colab or takign a copy of lora shared by the author) and a small set of data. If we fed this model more data and trained it for longer we may be able to achieve even better results!

# Conclusion

As seen that different weights provide different results. We need to make sure we get the right training data and hyperparameters to get the expected results.






